# DF Multi-Seed Validation — CE vs EDL

**Purpose**: Verify CE and EDL performance stability across 3 random seeds.

**Setup**: DF task, ViT-Base backbone, 2 modes (CE, EDL) × 3 seeds (42, 123, 456) = 6 runs.

**Expected runtime**: ~2.5h on T4 x2 GPU.

**Outputs**: Per-run test metrics JSON + aggregate summary with mean±std and cross-seed consistency.

## 1. Environment Setup

In [ ]:
!pip install transformers scikit-learn openpyxl scipy -q

!rm -rf /kaggle/working/fluorosis_project
!git clone https://github.com/XiaoHG/fluorosis_project.git /kaggle/working/fluorosis_project

import os
assert os.path.isdir("/kaggle/working/fluorosis_project/06_Implementation"), \
    "Git clone failed! Check repo visibility."
print("Code cloned successfully.")

In [ ]:
# ---- Master Setup - run this cell first ----
import os, sys, json, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader

# Pre-download pretrained weights while internet is available
print("Downloading pretrained weights...")
from transformers import ViTModel
_ = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
print("ViT weights cached.")

# Paths
CODE_ROOT = "/kaggle/working/fluorosis_project/06_Implementation"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
OUTPUT_DIR = "/kaggle/working"
sys.path.insert(0, CODE_ROOT)

from data import create_dataloaders
from models import build_model
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
print(f"Device: {device} | GPU: {gpu_name}")
print(f"CUDA available: {torch.cuda.is_available()} | GPU count: {torch.cuda.device_count()}")
print("Setup complete.")

## 2. Training Function

Self-contained `train_one()` for DF task. Supports CE and EDL modes with configurable seed.

In [ ]:
# ---- Core training function ----
import os, sys, json, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

sys.path.insert(0, "/kaggle/working/fluorosis_project/06_Implementation")
from data import create_dataloaders
from models import build_model
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


@torch.no_grad()
def evaluate_model(model, dataloader):
    """Full evaluation over a dataloader."""
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None:
            all_alpha.append(alpha.cpu())
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha, dim=0) if all_alpha else None
    z = torch.cat(all_z, dim=0)
    targets = torch.cat(all_targets, dim=0)
    return compute_metrics(alpha, z, targets)


def train_one(task, data_root, output_dir, mode="ce",
              batch_size=32, epochs=100, lr_backbone=1e-4, lr_head=1e-3,
              warmup_epochs=5, weight_decay=0.05, seed=42):
    """
    Train one DF model with given mode and seed.
    Returns dict with test metrics + per-class metrics.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    # Data
    train_loader, val_loader, test_loader = create_dataloaders(
        data_root, task=task, batch_size=batch_size, num_workers=2)
    print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} "
          f"test={len(test_loader.dataset)}")

    # Model
    model = build_model(task=task, mode=mode)
    model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    model_type = type(model).__name__
    print(f"Model: {n_params:,} params | Mode: {mode} | Type: {model_type}")

    # Criterion
    if mode == "ce":
        class CELocal(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                loss = F.nll_loss(F.log_softmax(z, dim=-1), targets)
                return loss, {"stage": 0, "L_ce": loss.item()}
        criterion = CELocal()
    else:  # edl
        from losses.edl_loss import EDLLoss
        edl_fn = EDLLoss(num_classes=4, kl_lambda=0.1)
        class EDLLocal(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                loss = edl_fn(alpha, targets, epoch, epochs)
                return loss, {"stage": 0, "L_edl": loss.item()}
        criterion = EDLLocal()

    # Optimizer
    backbone_params, head_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (backbone_params if n.startswith("backbone") else head_params).append(p)

    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": lr_backbone},
        {"params": head_params, "lr": lr_head},
    ], weight_decay=weight_decay)

    warmup_sched = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine_sched = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched],
                             milestones=[warmup_epochs])

    # Training loop
    best_val_acc, best_state = 0.0, None
    history = []

    for epoch in range(epochs):
        model.train()
        epoch_losses = {"L_ce": 0.0, "L_edl": 0.0}
        epoch_stage, n_batches = 0, 0

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, loss_info = criterion(alpha, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_stage = loss_info.get("stage", 0)
            for k in epoch_losses:
                if k in loss_info:
                    epoch_losses[k] += loss_info[k]
            n_batches += 1

        scheduler.step()
        for k in epoch_losses:
            epoch_losses[k] /= max(n_batches, 1)

        val_metrics = evaluate_model(model, val_loader)
        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            best_state = copy.deepcopy(model.state_dict())

        history.append({
            "epoch": epoch, "stage": epoch_stage,
            "train_loss": epoch_losses,
            "val_metrics": {k: v for k, v in val_metrics.items()
                            if isinstance(v, (float, int)) and not k.startswith("evidence_class_")},
        })

        loss_key = "L_ce" if mode == "ce" else "L_edl"
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == epochs - 1:
            print(f"[E{epoch+1:3d}/{epochs}] {loss_key}={epoch_losses.get(loss_key, 0):.4f} | "
                  f"Acc={val_metrics['acc']:.4f} F1={val_metrics['macro_f1']:.4f} "
                  f"QWK={val_metrics['qwk']:.4f} ECE={val_metrics['ece']:.4f} "
                  f"Unim={val_metrics['pct_unimodal']:.2%} | U={val_metrics.get('mean_uncertainty', 0):.3f}")

    # Load best and test
    model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_loader)

    # Save
    torch.save({"model_state_dict": best_state, "test_metrics": test_metrics},
               os.path.join(output_dir, "best.pt"))

    # Flatten for JSON: keep scalar metrics + confusion matrix
    flat_metrics = {}
    for k, v in test_metrics.items():
        if isinstance(v, (float, int, bool)):
            flat_metrics[k] = v
        elif isinstance(v, np.ndarray):
            flat_metrics[k] = v.tolist()
    flat_metrics["best_val_acc"] = best_val_acc
    flat_metrics["n_params"] = n_params
    flat_metrics["seed"] = seed
    flat_metrics["mode"] = mode

    with open(os.path.join(output_dir, "test_metrics.json"), "w") as f:
        json.dump(flat_metrics, f, indent=2)
    with open(os.path.join(output_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print(f"\n[Test] Acc={test_metrics['acc']:.4f} F1={test_metrics['macro_f1']:.4f} "
          f"QWK={test_metrics['qwk']:.4f} ECE={test_metrics['ece']:.4f} "
          f"Unimodal={test_metrics['pct_unimodal']:.2%} U-ECE={test_metrics.get('u_ece', 0):.4f} "
          f"AUROC(u)={test_metrics.get('auroc_u', 0):.4f}")
    return flat_metrics

print("train_one() and evaluate_model() ready.")

## 3. Run Multi-Seed Experiments

2 modes (CE, EDL) × 3 seeds (42, 123, 456) = 6 runs.

In [ ]:
import os, json, time

OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"

MODES = ["ce", "edl"]
SEEDS = [42, 123, 456]

all_results = {}  # key: "{mode}_seed{seed}"

t_start = time.time()
for mode in MODES:
    for seed in SEEDS:
        run_name = f"{mode}_seed{seed}"
        print(f"\n{'='*60}")
        print(f"DF | Mode: {mode} | Seed: {seed}")
        print(f"{'='*60}")
        out_dir = os.path.join(OUTPUT_DIR, f"df_{run_name}")
        metrics = train_one("df", DATA_ROOT, out_dir, mode=mode, seed=seed)
        all_results[run_name] = metrics
        elapsed = (time.time() - t_start) / 60
        print(f"  Elapsed: {elapsed:.1f} min")

# Save all raw results
with open(os.path.join(OUTPUT_DIR, "multiseed_all_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\nAll {len(MODES) * len(SEEDS)} runs complete. Total time: {(time.time() - t_start) / 60:.1f} min")

## 4. Aggregate Summary

Compute mean±std per metric across seeds for each mode. Check cross-seed consistency.

In [ ]:
import json, os
import numpy as np

OUTPUT_DIR = "/kaggle/working"

# Load results
with open(os.path.join(OUTPUT_DIR, "multiseed_all_results.json")) as f:
    all_results = json.load(f)

MODES = ["ce", "edl"]
SEEDS = [42, 123, 456]

# Metrics to aggregate
SCALAR_METRICS = ["acc", "macro_f1", "qwk", "ece", "pct_unimodal", "u_ece", "auroc_u"]

print("="*80)
print("DF Multi-Seed Validation Summary")
print("="*80)

# Per-mode aggregation
summary = {}
for mode in MODES:
    mode_runs = [f"{mode}_seed{s}" for s in SEEDS]
    mode_data = {k: [] for k in SCALAR_METRICS}
    conf_matrices = []

    for run_name in mode_runs:
        if run_name not in all_results:
            print(f"WARNING: {run_name} missing from results!")
            continue
        r = all_results[run_name]
        for k in SCALAR_METRICS:
            if k in r:
                mode_data[k].append(r[k])
        if "confusion_matrix" in r:
            conf_matrices.append(np.array(r["confusion_matrix"]))

    agg = {}
    for k, vals in mode_data.items():
        if vals:
            agg[k] = {"mean": np.mean(vals), "std": np.std(vals), "values": vals}

    # Sum confusion matrices across seeds
    if conf_matrices:
        agg["confusion_matrix_sum"] = np.sum(conf_matrices, axis=0).tolist()

    summary[mode] = agg

# Print summary table
print(f"\n{'Metric':<18}", end="")
for mode in MODES:
    print(f" {mode:>16}", end="")
print()
print("-" * 52)

for metric in SCALAR_METRICS:
    print(f"{metric:<18}", end="")
    for mode in MODES:
        if metric in summary[mode]:
            s = summary[mode][metric]
            print(f" {s['mean']:7.4f}±{s['std']:.4f}", end="")
        else:
            print(f" {'N/A':>16}", end="")
    print()

# Cross-seed consistency: print per-seed values
print(f"\n{'='*80}")
print("Per-Seed Breakdown")
print(f"{'='*80}")
for mode in MODES:
    print(f"\n--- {mode.upper()} ---")
    print(f"{'Seed':<8} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'%Unim':>8} {'U-ECE':>8} {'AUROC(u)':>8}")
    print("-" * 72)
    for seed in SEEDS:
        run_name = f"{mode}_seed{seed}"
        if run_name in all_results:
            r = all_results[run_name]
            print(f"{seed:<8} {r.get('acc',0):8.4f} {r.get('macro_f1',0):8.4f} "
                  f"{r.get('qwk',0):8.4f} {r.get('ece',0):8.4f} "
                  f"{r.get('pct_unimodal',0):7.2%} {r.get('u_ece',0):8.4f} "
                  f"{r.get('auroc_u',0):8.4f}")

# Cross-seed consistency check: CV of key metrics
print(f"\n{'='*80}")
print("Cross-Seed Consistency (CV = std/mean, lower is better)")
print(f"{'='*80}")
print(f"{'Metric':<18}", end="")
for mode in MODES:
    print(f" {mode:>12}", end="")
print()
print("-" * 44)
for metric in ["acc", "macro_f1", "qwk"]:
    print(f"{metric:<18}", end="")
    for mode in MODES:
        if metric in summary[mode]:
            s = summary[mode][metric]
            cv = s["std"] / s["mean"] * 100 if s["mean"] > 0 else float('inf')
            print(f" {cv:7.2f}%", end="        ")
    print()

# Confusion matrix (summed across seeds)
for mode in MODES:
    if "confusion_matrix_sum" in summary[mode]:
        cm = np.array(summary[mode]["confusion_matrix_sum"])
        print(f"\n{mode.upper()} — Confusion Matrix (sum across {len(SEEDS)} seeds):")
        labels = ["Normal", "Mild", "Moderate", "Severe"]
        print(f"{'':>10}", end="")
        for l in labels:
            print(f" {l:>8}", end="")
        print()
        for i, row in enumerate(cm):
            print(f"{labels[i]:>10}", end="")
            for v in row:
                print(f" {int(v):8d}", end="")
            print()

# Save summary
summary_serializable = {}
for mode in MODES:
    summary_serializable[mode] = {}
    for k, v in summary[mode].items():
        if isinstance(v, dict):
            summary_serializable[mode][k] = {
                "mean": float(v["mean"]),
                "std": float(v["std"]),
                "values": [float(x) for x in v["values"]]
            }
        elif isinstance(v, np.ndarray):
            summary_serializable[mode][k] = v.tolist()
        else:
            summary_serializable[mode][k] = v

with open(os.path.join(OUTPUT_DIR, "multiseed_summary.json"), "w") as f:
    json.dump(summary_serializable, f, indent=2)

print(f"\nSummary saved to {OUTPUT_DIR}/multiseed_summary.json")
print("\n=== MULTI-SEED VALIDATION COMPLETE ===")
print("Download /kaggle/working/multiseed_*.json for results.")